# Advanced Pipeline
Query understanding + adaptive retrieval + verification + LLM-as-judge evaluation.

In [1]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/multimodal_rag"))
from configs import config
from src.utils import setup_java
setup_java()

from src.retrieval import load_bm25, load_text_model, load_reranker, load_faiss_text, load_metadata
load_bm25(); load_text_model(); load_reranker(); load_faiss_text(); load_metadata()
print("All retrieval models loaded.")


[23:01:17] INFO utils -- JAVA_HOME=/home/aakul001/.conda/envs/rag310/lib/jvm
/home/aakul001/.conda/envs/rag310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[23:01:40] INFO retrieval -- Loading BM25 index: /home/aakul001/multimodal_rag/indexes/bm25_index

2026-04-28 23:01:40,624 - retrieval - INFO - Loading BM25 index: /home/aakul001/multimodal_rag/indexes/bm25_index
Apr 28, 2026 11:01:40 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
[23:01:40] INFO retrieval -- BM25 ready

2026-04-28 23:01:40,864 - retrieval - INFO - BM25 ready
[23:01:42] INFO retrieval -- Loading text embedding model: FremyCompany/BioLORD-2023-C

2026-04-28 23:01:42,954 - retr

All retrieval models loaded.


## Query Understanding Demo


In [2]:
from src.query_understanding import classify_query

test_queries = [
    "What does a chest X-ray look like in COVID-19?",
    "How is hypertension treated in primary care?",
    "Compare BM25 vs dense retrieval for biomedical search",
    "What is the mechanism of CRISPR-Cas9?",
]

print("=" * 65)
for q in test_queries:
    result = classify_query(q)
    print(f"\nQuery: {q}")
    print(f"  Type:     {result['type']}")
    print(f"  Weights:  BM25={result['weights']['bm25']} | Dense={result['weights']['dense_text']} | Image={result['weights']['dense_image']}")
    print(f"  Entities: {result['entities']}")
    print(f"  Reason:   {result['explanation']}")
print("=" * 65)


[23:02:35] INFO query_understanding -- Query type: [visual] confidence=0.087 | weights={'bm25': 0.2, 'dense_text': 0.3, 'dense_image': 0.5} | entities=['diseases']



2026-04-28 23:02:35,167 - query_understanding - INFO - Query type: [visual] confidence=0.087 | weights={'bm25': 0.2, 'dense_text': 0.3, 'dense_image': 0.5} | entities=['diseases']
[23:02:35] INFO query_understanding -- Query type: [general] confidence=0.000 | weights={'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15} | entities=['diseases']

2026-04-28 23:02:35,691 - query_understanding - INFO - Query type: [general] confidence=0.000 | weights={'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15} | entities=['diseases']
[23:02:35] INFO query_understanding -- Query type: [comparative] confidence=0.143 | weights={'bm25': 0.35, 'dense_text': 0.55, 'dense_image': 0.1} | entities=[]

2026-04-28 23:02:35,692 - query_understanding - INFO - Query type: [comparative] confidence=0.143 | weights={'bm25': 0.35, 'dense_text': 0.55, 'dense_image': 0.1} | entities=[]
[23:02:35] INFO query_understanding -- Query type: [factual] confidence=0.133 | weights={'bm25': 0.45, 'dense_text': 0.5, 'dense_


Query: What does a chest X-ray look like in COVID-19?
  Type:     visual
  Weights:  BM25=0.2 | Dense=0.3 | Image=0.5
  Entities: {'diseases': ['covid']}
  Reason:   Detected as 'visual' query. Adjusting weights: BM25=0.2, Dense=0.3, Image=0.5.

Query: How is hypertension treated in primary care?
  Type:     general
  Weights:  BM25=0.35 | Dense=0.5 | Image=0.15
  Entities: {'diseases': ['hypertension']}
  Reason:   General biomedical query. Using balanced retrieval weights.

Query: Compare BM25 vs dense retrieval for biomedical search
  Type:     comparative
  Weights:  BM25=0.35 | Dense=0.55 | Image=0.1
  Entities: {}
  Reason:   Detected as 'comparative' query. Adjusting weights: BM25=0.35, Dense=0.55, Image=0.1.

Query: What is the mechanism of CRISPR-Cas9?
  Type:     factual
  Weights:  BM25=0.45 | Dense=0.5 | Image=0.05
  Entities: {}
  Reason:   Detected as 'factual' query. Adjusting weights: BM25=0.45, Dense=0.5, Image=0.05.


## Adaptive Retrieval


In [3]:
from src.retrieval import retrieve
from src.query_understanding import classify_query
from src.utils import load_jsonl
from configs.config import QUERIES_FILE

queries = load_jsonl(QUERIES_FILE)
demo_q  = queries[0]

analysis = classify_query(demo_q["query"])
print(f"Query: {demo_q['query']}")
print(f"Detected type: {analysis['type']}")
print(f"Using weights: {analysis['weights']}\n")

results = retrieve(demo_q["query"], top_k=10)
print("Top 5 results:")
for _, row in results.head(5).iterrows():
    print(f"  [Rank {row['rank']} | {row['modality']}] {str(row['contents'])[:120]}...")


[23:02:42] INFO query_understanding -- Query type: [general] confidence=0.000 | weights={'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15} | entities=['diseases', 'procedures']

2026-04-28 23:02:42,944 - query_understanding - INFO - Query type: [general] confidence=0.000 | weights={'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15} | entities=['diseases', 'procedures']


Query: COVID-19 impact on healthcare systems
Detected type: general
Using weights: {'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15}



[23:02:43] INFO retrieval -- Loading image FAISS index...

2026-04-28 23:02:43,752 - retrieval - INFO - Loading image FAISS index...
[23:02:43] INFO retrieval -- Image FAISS ready: 5,000 vectors

2026-04-28 23:02:43,766 - retrieval - INFO - Image FAISS ready: 5,000 vectors
[23:02:44] INFO retrieval -- Loading BiomedCLIP via open_clip...

2026-04-28 23:02:44,190 - retrieval - INFO - Loading BiomedCLIP via open_clip...
[23:02:47] INFO retrieval -- BiomedCLIP ready on cuda

2026-04-28 23:02:47,618 - retrieval - INFO - BiomedCLIP ready on cuda


Top 5 results:
  [Rank 1 | text] Health system impact of COVID-19 on urban slum population of Bangladesh: a mixed-method rapid assessment study Objective...
  [Rank 2 | text] together (technical expert, KII-19). Online supplemental table 1 presents an overview of the opinions of all the technic...
  [Rank 3 | text] Further research is needed to assess the long-term impacts of COVID-19 on primary healthcare delivery. Introduction Prim...
  [Rank 4 | pdf] COVID-19 pandemic in Sweden. Supplementary Information The online version contains supplementary material available at 1...
  [Rank 5 | text] Impacts of the COVID-19 Pandemic on Primary Care Utilization: An Analysis of Primary Care Claims Data in Alberta, Canada...


## Advanced Generation with Structured Reasoning

In [4]:
from src.generation import generate_answer

result = generate_answer(
    query        = demo_q["query"],
    retrieval_df = results,
    query_type   = analysis["type"],
    verify       = False,
)

print(f"Query type used: {result['query_type']}")
print(f"Modalities retrieved: {result['modalities_used']}")
print(f"\nAnswer:\n{result['answer']}")


[23:02:55] INFO generation -- Loading LLM: meta-llama/Llama-3.2-3B-Instruct | device=cuda

2026-04-28 23:02:55,557 - generation - INFO - Loading LLM: meta-llama/Llama-3.2-3B-Instruct | device=cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 156.68it/s]
[23:02:59] INFO generation -- LLM ready

2026-04-28 23:02:59,977 - generation - INFO - LLM ready
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Query type used: general
Modalities retrieved: ['text', 'pdf']

Answer:
The COVID-19 pandemic has had a significant impact on healthcare systems worldwide, including strain on health services and disruptions to care delivery.


## Faithfulness Verification

In [5]:
from src.verification import verify_answer, format_verification_report

verification = verify_answer(
    answer   = result["answer"],
    passages = result["passages_used"],
)

print(format_verification_report(verification))


[23:03:17] INFO verification -- Loading NLI model: cross-encoder/nli-deberta-v3-small

2026-04-28 23:03:17,392 - verification - INFO - Loading NLI model: cross-encoder/nli-deberta-v3-small
Loading weights: 100%|██████████| 106/106 [00:00<00:00, 1027.73it/s]
[23:03:38] INFO verification -- NLI model ready on cuda

2026-04-28 23:03:38,747 - verification - INFO - NLI model ready on cuda
[23:03:40] INFO verification -- Verification: 1/1 claims supported | score=1.00 | verdict=faithful

2026-04-28 23:03:40,381 - verification - INFO - Verification: 1/1 claims supported | score=1.00 | verdict=faithful


Faithfulness Score: 100%
Verdict: FAITHFUL
Claims: 1/1 supported

  [OK] The COVID-19 pandemic has had a significant impact on healthcare systems worldwide, including strain


## Full Pipeline on All Queries

In [8]:
from src.generation import generate_batch
from src.retrieval import retrieve
import json
from configs.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_results = generate_batch(
    queries     = queries,
    retrieval_fn= retrieve,
    n_passages  = 5,
    top_k       = 10,
    verify      = True,
    use_query_understanding = True,
)

print("\n" + "=" * 60)
for r in all_results:
    v = r.get("verification", {})
    print(f"\n[{r['qid']}] {r['query']}")
    print(f"  Type: {r['query_type']} | Modalities: {r['modalities_used']}")
    print(f"  Faithfulness: {v.get('verdict','N/A')} ({v.get('overall_score',0):.0%})")
    print(f"  Answer: {r['answer'][:200]}...")
    print("-" * 60)

out = RESULTS_DIR / "advanced_rag_answers.json"
with open(out, "w") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=True)
print(f"\nSaved: {out}")


[23:12:54] INFO generation -- Processing q01: COVID-19 impact on healthcare systems...

2026-04-28 23:12:54,794 - generation - INFO - Processing q01: COVID-19 impact on healthcare systems...
[23:12:54] INFO query_understanding -- Query type: [general] confidence=0.000 | weights={'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15} | entities=['diseases', 'procedures']

2026-04-28 23:12:54,796 - query_understanding - INFO - Query type: [general] confidence=0.000 | weights={'bm25': 0.35, 'dense_text': 0.5, 'dense_image': 0.15} | entities=['diseases', 'procedures']
[23:12:54] INFO generation --   Query type: general | General biomedical query. Using balanced retrieval weights.

2026-04-28 23:12:54,797 - generation - INFO -   Query type: general | General biomedical query. Using balanced retrieval weights.
[23:13:03] INFO verification -- Verification: 1/1 claims supported | score=1.00 | verdict=faithful

2026-04-28 23:13:03,846 - verification - INFO - Verification: 1/1 claims supported | 



[q01] COVID-19 impact on healthcare systems
  Type: general | Modalities: ['text', 'pdf']
  Faithfulness: faithful (100%)
  Answer: The COVID-19 pandemic has had a significant impact on healthcare systems worldwide, including strain on health services and disruptions to care delivery....
------------------------------------------------------------

[q02] digital health transformation in primary care
  Type: general | Modalities: ['text']
  Faithfulness: faithful (100%)
  Answer: Digital health transformation is crucial for primary care, enabling improved patient outcomes, enhanced customer experiences, and environmental sustainability [Source 1, Source 3]. The COVID-19 pandem...
------------------------------------------------------------

[q03] gut microbiota changes during viral infection
  Type: general | Modalities: ['text']
  Faithfulness: faithful (100%)
  Answer: Gut microbiota changes during viral infection can lead to inflammation and immune system response, as reported in m

## LLM-as-Judge Evaluation

In [10]:
from src.llm_evaluation import evaluate_batch, print_evaluation_summary
import json
from configs.config import RESULTS_DIR

print("Running LLM-as-judge evaluation...")
evaluated = evaluate_batch(all_results, n_context=3)
print_evaluation_summary(evaluated)

out = RESULTS_DIR / "llm_judge_scores.json"
with open(out, "w") as f:
    json.dump(evaluated, f, indent=2, ensure_ascii=True)
print(f"\nSaved: {out}")


[23:16:16] INFO llm_evaluation -- Evaluating q01...

2026-04-28 23:16:16,050 - llm_evaluation - INFO - Evaluating q01...


Running LLM-as-judge evaluation...


[23:16:21] WARNING llm_evaluation -- Could not parse judge output: Expecting value: line 1 column 1 (char 0)

2026-04-28 23:16:21,942 - llm_evaluation - WARNING - Could not parse judge output: Expecting value: line 1 column 1 (char 0)
[23:16:21] INFO llm_evaluation -- LLM judge scores: faithfulness=0 relevance=0 overall=0

2026-04-28 23:16:21,945 - llm_evaluation - INFO - LLM judge scores: faithfulness=0 relevance=0 overall=0
[23:16:21] INFO llm_evaluation -- Evaluating q02...

2026-04-28 23:16:21,946 - llm_evaluation - INFO - Evaluating q02...
[23:16:27] INFO llm_evaluation -- LLM judge scores: faithfulness=5 relevance=5 overall=4.75

2026-04-28 23:16:27,221 - llm_evaluation - INFO - LLM judge scores: faithfulness=5 relevance=5 overall=4.75
[23:16:27] INFO llm_evaluation -- Evaluating q03...

2026-04-28 23:16:27,224 - llm_evaluation - INFO - Evaluating q03...
[23:16:32] WARNING llm_evaluation -- Could not parse judge output: Expecting value: line 1 column 1 (char 0)

2026-04-28 23:16:


  LLM-AS-JUDGE EVALUATION SUMMARY
  Query                               Faith   Rel  Comp  Cite   Avg
-----------------------------------------------------------------
  COVID-19 impact on healthcare syste     0     0     0     0   0.0
  digital health transformation in pr     5     5     5     5   4.8
  gut microbiota changes during viral     0     0     0     0   0.0
  chest X-ray findings in pneumonia       0     0     0     0   0.0
  hypertension treatment in primary c     5     5     5     5   4.8
  what are the benefits of telemedici     5     5     5     5   4.8
  MRI brain scan interpretation in ne     5     5     5     5   5.0
  how does COVID-19 affect lung tissu     4     5     4     4   4.0
  antibiotic resistance mechanisms in     0     0     0     0   0.0
  nurse leadership in digital healthc     0     0     0     0   0.0
-----------------------------------------------------------------
  AVERAGE                               2.4   2.5   2.4   2.4   2.3

Saved: /home/aak